#First prototype


In [1]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Constants
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 50
PANEL_WIDTH = 200
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE + MARGIN * 2 + PANEL_WIDTH
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2

# Colors
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_COLOR = (40, 40, 60)
BUTTON_COLOR = (70, 130, 200)

# Rewards
GOAL_REWARD = 100
HAZARD_PENALTY = -50
STEP_PENALTY = -1

# 10 different maze layouts (0=path, 1=wall, 2=hazard, 3=goal)
MAZE_LAYOUTS = [
    # Level 1
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 2, 0],
        [0, 0, 0, 0, 1, 3],
        [0, 1, 1, 0, 1, 0],
        [0, 2, 0, 0, 0, 0],
        [0, 1, 0, 1, 0, 0]
    ],
    # Level 2
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 2, 0],
        [0, 0, 0, 1, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [0, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 3
    [
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [2, 0, 1, 0, 0, 3],
        [0, 0, 1, 0, 2, 0]
    ],
    # Level 4
    [
        [0, 0, 1, 1, 0, 0],
        [1, 0, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 5
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [2, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 6
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ],
    # Level 7
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 8
    [
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 9
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 1, 0, 1],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 10
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ]
]

class MazeGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("10-Level Maze RL")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont('Arial', 16)
        self.big_font = pygame.font.SysFont('Arial', 24)
        
        self.current_level = 0
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.player_pos = [0, 0]
        self.goal_pos = self.find_goal()
        self.mode = "manual"  # or "training"
        self.running = True
        self.episode_count = 0
        self.max_episodes = 10000
        self.training_complete = False
        self.success_count = 0
        self.level_completion_stats = [0] * 10  # Track deaths per level before completion
        
        # Q-learning parameters
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))  # (x, y, action)
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.9
        self.min_epsilon = 0.1
        
        # UI buttons
        self.train_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 50, 160, 30)
        self.manual_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 90, 160, 30)
        self.reset_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 130, 160, 30)
    
    def find_goal(self):
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == 3:
                    return [x, y]
        return [0, 0]  # Fallback
    
    def reset(self):
        self.player_pos = [0, 0]
        return tuple(self.player_pos)
    
    def load_level(self, level):
        self.current_level = level
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.goal_pos = self.find_goal()
        self.reset()
    
    def step(self, action):
        x, y = self.player_pos
        new_x, new_y = x, y
        
        # 0=up, 1=right, 2=down, 3=left
        if action == 0 and x > 0: new_x = x - 1
        elif action == 1 and y < GRID_SIZE-1: new_y = y + 1
        elif action == 2 and x < GRID_SIZE-1: new_x = x + 1
        elif action == 3 and y > 0: new_y = y - 1
        
        # Check if move is valid (not a wall)
        if self.grid[new_x, new_y] != 1:
            self.player_pos = [new_x, new_y]
        
        # Check if player reached goal or hazard
        cell_value = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell_value == 3:  # Goal
            reward = GOAL_REWARD
            done = True
            self.success_count += 1
            
            # Move to next level or complete training if finished all levels
            if self.current_level < 9:
                self.load_level(self.current_level + 1)
                done = False  # Continue training
            else:
                self.training_complete = True
                print("Training complete! Level completion stats:")
                for i, deaths in enumerate(self.level_completion_stats):
                    print(f"Level {i+1}: {deaths} deaths before completion")
                
        elif cell_value == 2:  # Hazard
            reward = HAZARD_PENALTY
            done = True
            # Record death for current level
            self.level_completion_stats[self.current_level] += 1
            # Reset to level 1
            self.load_level(0)
        else:  # Path
            reward = STEP_PENALTY
            done = False
        
        if done:
            self.reset()
            self.episode_count += 1
            
            if self.episode_count >= self.max_episodes and not self.training_complete:
                self.training_complete = True
        
        return tuple(self.player_pos), reward, done
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])  # Best action
    
    def update_q_table(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        # Q-learning formula
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount_factor * best_next - self.q_table[x, y, action]
        )
    
    def train_episode(self):
        state = self.reset()
        done = False
        
        while not done:
            action = self.choose_action(state)
            next_state, reward, done = self.step(action)
            self.update_q_table(state, action, reward, next_state)
            state = next_state
        
        # Reduce exploration over time
        self.epsilon = max(self.min_epsilon, self.epsilon * 0.995)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == QUIT:
                self.running = False
            
            elif event.type == MOUSEBUTTONDOWN:
                if event.button == 1:  # Left click
                    mouse_pos = pygame.mouse.get_pos()
                    
                    if self.train_btn.collidepoint(mouse_pos):
                        self.mode = "training"
                    elif self.manual_btn.collidepoint(mouse_pos):
                        self.mode = "manual"
                    elif self.reset_btn.collidepoint(mouse_pos):
                        self.load_level(0)
                        self.episode_count = 0
                        self.success_count = 0
                        self.training_complete = False
                        self.level_completion_stats = [0] * 10
                        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                        self.epsilon = 0.9
            
            elif event.type == KEYDOWN and self.mode == "manual":
                if event.key == K_UP: self.step(0)
                elif event.key == K_RIGHT: self.step(1)
                elif event.key == K_DOWN: self.step(2)
                elif event.key == K_LEFT: self.step(3)
    
    def update(self):
        if self.mode == "training" and not self.training_complete:
            self.train_episode()
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Draw grid
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Cell color
                if self.grid[x, y] == 1:  # Wall
                    pygame.draw.rect(self.screen, WALL_COLOR, rect)
                elif self.grid[x, y] == 2:  # Hazard
                    pygame.draw.rect(self.screen, HAZARD_COLOR, rect)
                elif self.grid[x, y] == 3:  # Goal
                    pygame.draw.rect(self.screen, GOAL_COLOR, rect)
                
                # Grid lines
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Player
                if x == self.player_pos[0] and y == self.player_pos[1]:
                    pygame.draw.circle(
                        self.screen, PLAYER_COLOR,
                        (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                         MARGIN + x * CELL_SIZE + CELL_SIZE//2),
                        CELL_SIZE//3
                    )
        
        # Draw info panel
        panel = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN, 0, PANEL_WIDTH, WINDOW_HEIGHT)
        pygame.draw.rect(self.screen, PANEL_COLOR, panel)
        
        # Draw buttons
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.train_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.manual_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.reset_btn)
        
        # Draw button text
        train_text = self.font.render("Train Mode", True, TEXT_COLOR)
        manual_text = self.font.render("Manual Mode", True, TEXT_COLOR)
        reset_text = self.font.render("Reset Training", True, TEXT_COLOR)
        
        self.screen.blit(train_text, (self.train_btn.x + 10, self.train_btn.y + 5))
        self.screen.blit(manual_text, (self.manual_btn.x + 10, self.manual_btn.y + 5))
        self.screen.blit(reset_text, (self.reset_btn.x + 10, self.reset_btn.y + 5))
        
        # Draw stats
        mode_text = self.font.render(f"Mode: {self.mode}", True, TEXT_COLOR)
        episode_text = self.font.render(f"Episodes: {self.episode_count}/{self.max_episodes}", True, TEXT_COLOR)
        level_text = self.font.render(f"Level: {self.current_level + 1}/10", True, TEXT_COLOR)
        
        self.screen.blit(mode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 180))
        self.screen.blit(episode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 210))
        self.screen.blit(level_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 240))
        
        # Training complete message
        if self.training_complete:
            complete_text = self.big_font.render("Training Complete!", True, (50, 180, 80))
            result_text = self.font.render(f"Level completion stats:", True, TEXT_COLOR)
            
            self.screen.blit(complete_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 300))
            self.screen.blit(result_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 340))
            
            # Display level completion stats
            for i in range(min(5, 10)):  # First 5 levels
                stat_text = self.font.render(
                    f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                    True, TEXT_COLOR
                )
                self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 370 + i*30))
            
            if len(self.level_completion_stats) > 5:  # Next 5 levels if they exist
                for i in range(5, 10):
                    stat_text = self.font.render(
                        f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                        True, TEXT_COLOR
                    )
                    self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 120, MARGIN + 370 + (i-5)*30))
        
        pygame.display.flip()
    
    def run(self):
        while self.running:
            self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = MazeGame()
    game.run()

pygame 2.6.1 (SDL 2.28.4, Python 3.10.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
Training complete! Level completion stats:
Level 1: 41 deaths before completion
Level 2: 16 deaths before completion
Level 3: 3 deaths before completion
Level 4: 2 deaths before completion
Level 5: 5 deaths before completion
Level 6: 0 deaths before completion
Level 7: 1 deaths before completion
Level 8: 1 deaths before completion
Level 9: 1 deaths before completion
Level 10: 1 deaths before completion
Training complete! Level completion stats:
Level 1: 34 deaths before completion
Level 2: 16 deaths before completion
Level 3: 13 deaths before completion
Level 4: 6 deaths before completion
Level 5: 5 deaths before completion
Level 6: 1 deaths before completion
Level 7: 0 deaths before completion
Level 8: 1 deaths before completion
Level 9: 0 deaths before completion
Level 10: 1 deaths before completion


KeyboardInterrupt: 

In [8]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Constants
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 50
PANEL_WIDTH = 200
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE + MARGIN * 2 + PANEL_WIDTH
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2

# Colors
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_COLOR = (40, 40, 60)
BUTTON_COLOR = (70, 130, 200)
ACTION_COLOR = (180, 180, 100)
VISITED_COLOR = (80, 80, 120)

# Rewards
GOAL_REWARD = 100
HAZARD_PENALTY = -50
STEP_PENALTY = -1

# 10 different maze layouts (0=path, 1=wall, 2=hazard, 3=goal)
MAZE_LAYOUTS = [
    # Level 1
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 2, 0],
        [0, 0, 0, 0, 1, 3],
        [0, 1, 1, 0, 1, 0],
        [0, 2, 0, 0, 0, 0],
        [0, 1, 0, 1, 0, 0]
    ],
    # Level 2
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 2, 0],
        [0, 0, 0, 1, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [0, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 3
    [
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [2, 0, 1, 0, 0, 3],
        [0, 0, 1, 0, 2, 0]
    ],
    # Level 4
    [
        [0, 0, 1, 1, 0, 0],
        [1, 0, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 5
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [2, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 6
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ],
    # Level 7
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 8
    [
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 9
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 1, 0, 1],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 10
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ]
]

class MazeGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("10-Level Maze RL - Visual Training")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont('Arial', 16)
        self.big_font = pygame.font.SysFont('Arial', 24)
        
        self.current_level = 0
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.player_pos = [0, 0]
        self.goal_pos = self.find_goal()
        self.mode = "manual"  # or "training"
        self.running = True
        self.episode_count = 0
        self.max_episodes = 10000
        self.training_complete = False
        self.success_count = 0
        self.level_completion_stats = [0] * 10  # Track deaths per level before completion
        self.visited_positions = set()  # Track visited positions
        self.last_action = None
        self.last_reward = 0
        self.paused = False
        
        # Q-learning parameters
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))  # (x, y, action)
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.9
        self.min_epsilon = 0.1
        
        # UI buttons
        self.train_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 50, 160, 30)
        self.manual_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 90, 160, 30)
        self.reset_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 130, 160, 30)
        self.pause_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 170, 160, 30)
    
    def find_goal(self):
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == 3:
                    return [x, y]
        return [0, 0]  # Fallback
    
    def reset(self):
        self.player_pos = [0, 0]
        self.visited_positions = set()
        self.visited_positions.add((0, 0))
        self.last_action = None
        self.last_reward = 0
        return tuple(self.player_pos)
    
    def load_level(self, level):
        self.current_level = level
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.goal_pos = self.find_goal()
        self.reset()
    
    def step(self, action):
        x, y = self.player_pos
        new_x, new_y = x, y
        self.last_action = action
        
        # 0=up, 1=right, 2=down, 3=left
        if action == 0 and x > 0: new_x = x - 1
        elif action == 1 and y < GRID_SIZE-1: new_y = y + 1
        elif action == 2 and x < GRID_SIZE-1: new_x = x + 1
        elif action == 3 and y > 0: new_y = y - 1
        
        # Check if move is valid (not a wall)
        if self.grid[new_x, new_y] != 1:
            self.player_pos = [new_x, new_y]
            self.visited_positions.add((new_x, new_y))
        
        # Check if player reached goal or hazard
        cell_value = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell_value == 3:  # Goal
            reward = GOAL_REWARD
            done = True
            self.success_count += 1
            
            # Move to next level or complete training if finished all levels
            if self.current_level < 9:
                self.load_level(self.current_level + 1)
                done = False  # Continue training
            else:
                self.training_complete = True
                print("Training complete! Level completion stats:")
                for i, deaths in enumerate(self.level_completion_stats):
                    print(f"Level {i+1}: {deaths} deaths before completion")
                
        elif cell_value == 2:  # Hazard
            reward = HAZARD_PENALTY
            done = True
            # Record death for current level
            self.level_completion_stats[self.current_level] += 1
            # Reset to level 1
            self.load_level(0)
        else:  # Path
            reward = STEP_PENALTY
            done = False
        
        self.last_reward = reward
        
        if done:
            self.reset()
            self.episode_count += 1
            
            if self.episode_count >= self.max_episodes and not self.training_complete:
                self.training_complete = True
        
        return tuple(self.player_pos), reward, done
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])  # Best action
    
    def update_q_table(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        # Q-learning formula
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount_factor * best_next - self.q_table[x, y, action]
        )
    
    def train_step(self):
        if not hasattr(self, 'current_state'):
            self.current_state = self.reset()
        
        action = self.choose_action(self.current_state)
        next_state, reward, done = self.step(action)
        self.update_q_table(self.current_state, action, reward, next_state)
        self.current_state = next_state
        
        # Reduce exploration over time
        self.epsilon = max(self.min_epsilon, self.epsilon * 0.9995)
        
        return done
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == QUIT:
                self.running = False
            
            elif event.type == MOUSEBUTTONDOWN:
                if event.button == 1:  # Left click
                    mouse_pos = pygame.mouse.get_pos()
                    
                    if self.train_btn.collidepoint(mouse_pos):
                        self.mode = "training"
                        self.paused = False
                    elif self.manual_btn.collidepoint(mouse_pos):
                        self.mode = "manual"
                        self.paused = False
                    elif self.reset_btn.collidepoint(mouse_pos):
                        self.load_level(0)
                        self.episode_count = 0
                        self.success_count = 0
                        self.training_complete = False
                        self.level_completion_stats = [0] * 10
                        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                        self.epsilon = 0.9
                        if hasattr(self, 'current_state'):
                            del self.current_state
                    elif self.pause_btn.collidepoint(mouse_pos) and self.mode == "training":
                        self.paused = not self.paused
            
            elif event.type == KEYDOWN and self.mode == "manual":
                if event.key == K_UP: self.step(0)
                elif event.key == K_RIGHT: self.step(1)
                elif event.key == K_DOWN: self.step(2)
                elif event.key == K_LEFT: self.step(3)
                elif event.key == K_SPACE and self.mode == "training":
                    self.paused = not self.paused
    
    def update(self):
        if self.mode == "training" and not self.training_complete and not self.paused:
            self.train_step()
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Draw grid
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Cell color - visited cells are darker
                if (x, y) in self.visited_positions:
                    pygame.draw.rect(self.screen, VISITED_COLOR, rect)
                
                if self.grid[x, y] == 1:  # Wall
                    pygame.draw.rect(self.screen, WALL_COLOR, rect)
                elif self.grid[x, y] == 2:  # Hazard
                    pygame.draw.rect(self.screen, HAZARD_COLOR, rect)
                elif self.grid[x, y] == 3:  # Goal
                    pygame.draw.rect(self.screen, GOAL_COLOR, rect)
                
                # Grid lines
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Player
                if x == self.player_pos[0] and y == self.player_pos[1]:
                    pygame.draw.circle(
                        self.screen, PLAYER_COLOR,
                        (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                         MARGIN + x * CELL_SIZE + CELL_SIZE//2),
                        CELL_SIZE//3
                    )
        
        # Draw action indicator
        if self.last_action is not None and self.mode == "training":
            x, y = self.player_pos
            arrow_pos = (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                        MARGIN + x * CELL_SIZE + CELL_SIZE//2)
            
            # Draw arrow for last action
            if self.last_action == 0:  # Up
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0], arrow_pos[1] - 15),
                    (arrow_pos[0] - 10, arrow_pos[1] + 5),
                    (arrow_pos[0] + 10, arrow_pos[1] + 5)
                ])
            elif self.last_action == 1:  # Right
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0] + 15, arrow_pos[1]),
                    (arrow_pos[0] - 5, arrow_pos[1] - 10),
                    (arrow_pos[0] - 5, arrow_pos[1] + 10)
                ])
            elif self.last_action == 2:  # Down
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0], arrow_pos[1] + 15),
                    (arrow_pos[0] - 10, arrow_pos[1] - 5),
                    (arrow_pos[0] + 10, arrow_pos[1] - 5)
                ])
            elif self.last_action == 3:  # Left
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0] - 15, arrow_pos[1]),
                    (arrow_pos[0] + 5, arrow_pos[1] - 10),
                    (arrow_pos[0] + 5, arrow_pos[1] + 10)
                ])
        
        # Draw info panel
        panel = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN, 0, PANEL_WIDTH, WINDOW_HEIGHT)
        pygame.draw.rect(self.screen, PANEL_COLOR, panel)
        
        # Draw buttons
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.train_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.manual_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.reset_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.pause_btn)
        
        # Draw button text
        train_text = self.font.render("Train Mode", True, TEXT_COLOR)
        manual_text = self.font.render("Manual Mode", True, TEXT_COLOR)
        reset_text = self.font.render("Reset Training", True, TEXT_COLOR)
        pause_text = self.font.render("Pause/Resume", True, TEXT_COLOR)
        
        self.screen.blit(train_text, (self.train_btn.x + 10, self.train_btn.y + 5))
        self.screen.blit(manual_text, (self.manual_btn.x + 10, self.manual_btn.y + 5))
        self.screen.blit(reset_text, (self.reset_btn.x + 10, self.reset_btn.y + 5))
        self.screen.blit(pause_text, (self.pause_btn.x + 10, self.pause_btn.y + 5))
        
        # Draw stats
        mode_text = self.font.render(f"Mode: {self.mode}", True, TEXT_COLOR)
        episode_text = self.font.render(f"Episodes: {self.episode_count}/{self.max_episodes}", True, TEXT_COLOR)
        level_text = self.font.render(f"Level: {self.current_level + 1}/10", True, TEXT_COLOR)
        epsilon_text = self.font.render(f"Exploration: {self.epsilon:.2f}", True, TEXT_COLOR)
        reward_text = self.font.render(f"Last reward: {self.last_reward}", True, TEXT_COLOR)
        action_text = self.font.render(f"Last action: {['Up', 'Right', 'Down', 'Left'][self.last_action] if self.last_action is not None else 'None'}", True, TEXT_COLOR)
        
        self.screen.blit(mode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 220))
        self.screen.blit(episode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 250))
        self.screen.blit(level_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 280))
        self.screen.blit(epsilon_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 310))
        self.screen.blit(reward_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 340))
        self.screen.blit(action_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 370))
        
        # Pause indicator
        if self.paused and self.mode == "training":
            pause_indicator = self.big_font.render("PAUSED", True, (220, 80, 60))
            self.screen.blit(pause_indicator, (GRID_SIZE*CELL_SIZE + MARGIN + 50, MARGIN + 400))
        
        # Training complete message
        if self.training_complete:
            complete_text = self.big_font.render("Training Complete!", True, (50, 180, 80))
            result_text = self.font.render(f"Level completion stats:", True, TEXT_COLOR)
            
            self.screen.blit(complete_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 450))
            self.screen.blit(result_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 480))
            
            # Display level completion stats
            for i in range(min(5, 10)):  # First 5 levels
                stat_text = self.font.render(
                    f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                    True, TEXT_COLOR
                )
                self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 510 + i*30))
            
            if len(self.level_completion_stats) > 5:  # Next 5 levels if they exist
                for i in range(5, 10):
                    stat_text = self.font.render(
                        f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                        True, TEXT_COLOR
                    )
                    self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 120, MARGIN + 510 + (i-5)*30))
        
        pygame.display.flip()
    
    def run(self):
        while self.running:
            self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(60)  # Limit to 60 FPS
        
        pygame.quit()

if __name__ == "__main__":
    game = MazeGame()
    game.run()

In [3]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Constants
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 50
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2

# Colors
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
Q_VALUE_COLOR = (180, 180, 250)

# Rewards
GOAL_REWARD = 100
HAZARD_PENALTY = -50
STEP_PENALTY = -1

# Maze layout (0=path, 1=wall, 2=hazard, 3=goal)
MAZE_LAYOUT = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

class MazeGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Maze Q-Learning")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont('Arial', 16)
        
        self.grid = np.array(MAZE_LAYOUT)
        self.player_pos = [0, 0]
        self.goal_pos = self.find_goal()
        self.mode = "manual"  # or "training"
        self.running = True
        self.episode_count = 0
        
        # Q-learning parameters
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))  # (x, y, action)
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.9
        self.min_epsilon = 0.1
        self.training_speed = 10  # episodes per frame
    
    def find_goal(self):
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == 3:
                    return [x, y]
        return [0, 0]  # Fallback
    
    def reset(self):
        self.player_pos = [0, 0]
        return tuple(self.player_pos)
    
    def step(self, action):
        x, y = self.player_pos
        new_x, new_y = x, y
        
        # 0=up, 1=right, 2=down, 3=left
        if action == 0 and x > 0: new_x = x - 1
        elif action == 1 and y < GRID_SIZE-1: new_y = y + 1
        elif action == 2 and x < GRID_SIZE-1: new_x = x + 1
        elif action == 3 and y > 0: new_y = y - 1
        
        # Check if move is valid (not a wall)
        if self.grid[new_x, new_y] != 1:
            self.player_pos = [new_x, new_y]
        
        # Check if player reached goal or hazard
        cell_value = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell_value == 3:  # Goal
            reward = GOAL_REWARD
            done = True
        elif cell_value == 2:  # Hazard
            reward = HAZARD_PENALTY
            done = True
        else:  # Path
            reward = STEP_PENALTY
            done = False
        
        if done:
            self.reset()
            self.episode_count += 1
        
        return tuple(self.player_pos), reward, done
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])  # Best action
    
    def update_q_table(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        # Q-learning formula
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount_factor * best_next - self.q_table[x, y, action]
        )
    
    def train_episode(self):
        state = self.reset()
        done = False
        
        while not done:
            action = self.choose_action(state)
            next_state, reward, done = self.step(action)
            self.update_q_table(state, action, reward, next_state)
            state = next_state
        
        # Reduce exploration over time
        self.epsilon = max(self.min_epsilon, self.epsilon * 0.999)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == QUIT:
                self.running = False
            
            elif event.type == KEYDOWN:
                if event.key == K_ESCAPE:
                    self.running = False
                elif event.key == K_t:
                    self.mode = "training"
                elif event.key == K_m:
                    self.mode = "manual"
                elif event.key == K_r:
                    self.reset()
                    self.episode_count = 0
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                elif event.key == K_PLUS or event.key == K_EQUALS:
                    self.training_speed = min(100, self.training_speed + 5)
                elif event.key == K_MINUS:
                    self.training_speed = max(1, self.training_speed - 5)
                
                if self.mode == "manual":
                    if event.key == K_UP: self.step(0)
                    elif event.key == K_RIGHT: self.step(1)
                    elif event.key == K_DOWN: self.step(2)
                    elif event.key == K_LEFT: self.step(3)
    
    def update(self):
        if self.mode == "training":
            for _ in range(self.training_speed):
                self.train_episode()
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Draw main grid (environment)
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Cell color
                if self.grid[x, y] == 1:  # Wall
                    pygame.draw.rect(self.screen, WALL_COLOR, rect)
                elif self.grid[x, y] == 2:  # Hazard
                    pygame.draw.rect(self.screen, HAZARD_COLOR, rect)
                elif self.grid[x, y] == 3:  # Goal
                    pygame.draw.rect(self.screen, GOAL_COLOR, rect)
                
                # Grid lines
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Player
                if x == self.player_pos[0] and y == self.player_pos[1]:
                    pygame.draw.circle(
                        self.screen, PLAYER_COLOR,
                        (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                         MARGIN + x * CELL_SIZE + CELL_SIZE//2),
                        CELL_SIZE//3
                    )
        
        # Draw Q-value grid
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN * 2 + GRID_SIZE * CELL_SIZE + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Background
                pygame.draw.rect(self.screen, (40, 40, 60), rect)
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Q-values
                max_q = np.max(self.q_table[x, y])
                if max_q != 0:  # Only draw if we have values
                    q_text = self.font.render(f"{max_q:.1f}", True, Q_VALUE_COLOR)
                    self.screen.blit(q_text, (rect.x + 5, rect.y + 5))
        
        # Draw stats
        mode_text = self.font.render(f"Mode: {self.mode}", True, TEXT_COLOR)
        episode_text = self.font.render(f"Episodes: {self.episode_count}", True, TEXT_COLOR)
        epsilon_text = self.font.render(f"Epsilon: {self.epsilon:.2f}", True, TEXT_COLOR)
        speed_text = self.font.render(f"Speed: {self.training_speed}", True, TEXT_COLOR)
        controls_text = self.font.render("Controls: T=Train, M=Manual, R=Reset", True, TEXT_COLOR)
        controls_text2 = self.font.render("+/-=Speed, Arrows=Move", True, TEXT_COLOR)
        
        self.screen.blit(mode_text, (MARGIN, WINDOW_HEIGHT - MARGIN + 10))
        self.screen.blit(episode_text, (MARGIN, WINDOW_HEIGHT - MARGIN + 30))
        self.screen.blit(epsilon_text, (MARGIN, WINDOW_HEIGHT - MARGIN + 50))
        self.screen.blit(speed_text, (MARGIN + 200, WINDOW_HEIGHT - MARGIN + 10))
        self.screen.blit(controls_text, (MARGIN + 200, WINDOW_HEIGHT - MARGIN + 30))
        self.screen.blit(controls_text2, (MARGIN + 200, WINDOW_HEIGHT - MARGIN + 50))
        
        pygame.display.flip()
    
    def run(self):
        while self.running:
            self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = MazeGame()
    game.run()